In [ ]:
import os

In [ ]:
%pwd

In [ ]:
import torch

In [ ]:
import dagshub
dagshub.init(repo_owner='Nilansh-garg', repo_name='HIV_Inhibitor_Prediction_GNN_MLFlow-DVC', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

In [ ]:
from dataclasses import dataclass
from pathlib import Path

In [ ]:
@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path  # Path to your processed dataset
    all_params: dict
    mlflow_uri: str
    params_batch_size: int
    params_device: str



In [ ]:
from GNNClassfier.constants import *
from GNNClassfier.utils.common import read_yaml, create_directories, save_json

In [ ]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
        
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model= Path(self.config.training.trained_model_path),
            training_data=Path(self.config.training.test_data_path),
            mlflow_uri=str(self.config.evaluation.mlflow_uri),
            all_params=self.params,
            params_batch_size=self.params.BATCH_SIZE,
            params_device="cuda" if torch.cuda.is_available() else "cpu"
        )
        return eval_config



In [ ]:
import torch
from dataclasses import dataclass
from pathlib import Path
from torch_geometric.loader import DataLoader

In [ ]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config
        self.device = torch.device(self.config.params_device)
        self.score = None

    def _load_test_data(self):
        # Loading the .pt file containing the Test Dataset (PyG format)
        def load_graphs_from_dir(directory_path):
            processed_dir = Path(directory_path) / "processed"
            
            if not processed_dir.exists():
                raise FileNotFoundError(f"The directory {processed_dir} does not exist. Check your Stage 03 output.")
            
            # Find all .pt files and sort them to maintain consistency
            graph_files = sorted([str(f) for f in processed_dir.glob("*.pt")])
            
            if len(graph_files) == 0:
                raise ValueError(f"No .pt files found in {processed_dir}")
                
            print(f"Loading {len(graph_files)} graphs from {processed_dir}...")
            
            return [torch.load(f, weights_only = False) for f in graph_files]

        # 2. Load Data
# This function returns the list of graphs
        self.test_dataset = load_graphs_from_dir(self.config.training_data) 
        
        # Now just pass that list directly to the DataLoader
        self.test_loader = DataLoader(
            self.test_dataset, 
            batch_size=self.config.params_batch_size, 
            shuffle=False
        )

    def evaluation(self):
        # Load the PyTorch GNN model
        
        self.model = torch.load(self.config.base_model_path, weights_only=False)
        state_dict = torch.load(self.config.path_of_model)
        self.model.load_state_dict(state_dict)
        # 3. Now you can move it to the device
        self.model.to(self.device)
        self.model.eval()
        
        self._load_test_data()
        
        criterion = torch.nn.CrossEntropyLoss()
        total_loss = 0
        correct = 0

        with torch.no_grad():
            for data in self.test_loader:
                data = data.to(self.device)
                out = self.model(data.x, data.edge_index, data.batch)
                
                # 1. Calculate Loss
                # CrossEntropyLoss: out is [Batch, 2], target must be [Batch]
                loss = criterion(out, data.y.long().view(-1))
                total_loss += loss.item()
                
                # 2. Calculate Accuracy
                pred = out.argmax(dim=1) # Get the index of the max logit
                correct += (pred == data.y.view(-1)).sum().item()

        # Final Metrics
        avg_loss = total_loss / len(self.test_loader)
        accuracy = correct / len(self.test_dataset)

        self.score = [avg_loss, accuracy]

    def save_score(self):
        
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path(self.config.score_util_path), data=scores)

    def log_into_mlflow(self):
        mlflow.set_tracking_uri(self.config.mlflow_uri)
        mlflow.set_experiment("HIV-Inhibitor-GNN-Evaluation")

        with mlflow.start_run(run_name="evaluation_run"):
            # Log all parameters from params.yaml
            mlflow.log_params(self.config.all_params)

            # Log Metrics
            mlflow.log_metrics({
                "test_loss": self.score[0],
                "test_accuracy": self.score[1]
            })

            # Log Model using the PyTorch flavor
            if self.model:
                mlflow.pytorch.log_model(
                    self.model, 
                    artifact_path="model",
                    registered_model_name="HIV_GAT_Model"
                )

In [ ]:
# Execution Pipeline
if __name__ == "__main__":
    try:
        config_manager = ConfigurationManager()
        eval_config = config_manager.get_evaluation_config()
        evaluation = Evaluation(config=eval_config)
        evaluation.evaluation()
        evaluation.save_score()
        evaluation.log_into_mlflow()
    except Exception as e:
        raise e